In [31]:
import pandas as pd

inventory_df = pd.read_csv('../external/inventory.csv', sep='|')

## Column Names in the Inventory DataFrame

The inventory dataset contains the following columns:

- **vin**: Vehicle Identification Number
- **year**: Year the vehicle was manufactured
- **make**: Vehicle manufacturer/brand
- **model**: Vehicle model
- **trim**: Vehicle trim level
- **dealer_name**: Name of the dealership
- **dealer_street**: Street address of dealer
- **dealer_city**: City where dealer is located
- **dealer_state**: State where dealer is located
- **dealer_zip**: Dealer ZIP code
- **listing_price**: Vehicle's listed price
- **listing_mileage**: Vehicle's mileage
- **used**: Whether the vehicle is used (True) or new (False)
- **certified**: Whether the vehicle is certified pre-owned
- **style**: Vehicle body style
- **driven_wheels**: Type of wheel drive (FWD, RWD, 4WD, AWD)
- **engine**: Engine specifications
- **fuel_type**: Type of fuel used
- **exterior_color**: Vehicle's exterior color
- **interior_color**: Vehicle's interior color
- **seller_website**: Website URL of the seller
- **first_seen_date**: Date when listing was first seen
- **last_seen_date**: Date when listing was last seen
- **dealer_vdp_last_seen_date**: Last date the dealer's vehicle detail page was seen
- **listing_status**: Status of the vehicle listing

In [32]:
# Rename potentially conflicting column names to be more descriptive
inventory_df = inventory_df.rename(columns={
  'model': 'vehicle_model',
  'trim' : 'vehicle_trim',
  'style': 'body_style',
  'year': 'model_year',
  'make': 'manufacturer',
  'used': 'is_used',
  'certified': 'is_certified'
})

# Display the first few rows with new column names
inventory_df.head()

,vin,model_year,manufacturer,vehicle_model,vehicle_trim,dealer_name,dealer_street,dealer_city,dealer_state,dealer_zip,...,driven_wheels,engine,fuel_type,exterior_color,interior_color,seller_website,first_seen_date,last_seen_date,dealer_vdp_last_seen_date,listing_status
0,KNAFK4A61F5428652,2015,Kia,FORTE,LX,X Nation Auto Group,6003 Bandera Rd,San Antonio,TX,78238,...,FWD,1.8L,NaN,Silver,Gray,https://xnationautogroup.com,2021-11-24,2022-08-17,2022-08-17,NaN
1,3GTP1NEJ7HG400459,2017,GMC,Sierra 1500,1500 SLT,X Nation Auto Group,6003 Bandera Rd,San Antonio,TX,78238,...,4X2,6.2L V8,NaN,White,N/a,https://xnationautogroup.com,2022-08-04,2022-08-17,2022-08-17,NaN
2,1B7HF13Z1XJ596452,1999,Dodge,Ram Pickup 1500,Laramie SLT,Clear Choice Automotive,3815 SE Naef Rd,Milwaukie,OR,97267,...,4WD,5.9L V8,Gasoline,Silver,Gray,https://www.clearchoiceautomotive.com,2022-05-10,2022-08-17,2022-05-11,NaN
3,JTEGD21A720036451,2002,Toyota,Highlander,4DR 2WD AT,Max Auto Sales,4895 Johnston St,Lafayette,LA,70503,...,NaN,NaN,NaN,NaN,NaN,https://maxautosalesla.com,2022-07-14,2022-08-17,2022-08-17,NaN
4,1NXBU4EE8AZ361720,2010,Toyota,Corolla,NaN,Max Auto Sales,4895 Johnston St,Lafayette,LA,70503,...,NaN,NaN,NaN,NaN,NaN,https://maxautosalesla.com,2022-07-26,2022-08-17,2022-08-17,NaN


In [33]:
import numpy as np

# Display the count of missing values before cleaning
print("Before cleaning:")
print(f"Missing values in listing_price: {inventory_df['listing_price'].isna().sum()}")
print(f"Missing values in listing_mileage: {inventory_df['listing_mileage'].isna().sum()}")

# Check for and clean potential data issues:
# - Convert negative values to NaN as they don't make sense for price and mileage
# - Very high mileage values that may be data entry errors
inventory_df.loc[inventory_df['listing_price'] < 0, 'listing_price'] = np.nan
inventory_df.loc[inventory_df['listing_mileage'] < 0, 'listing_mileage'] = np.nan
inventory_df.loc[inventory_df['listing_mileage'] > 1000000, 'listing_mileage'] = np.nan

# For used vehicles, zero price is suspicious and likely a missing value
inventory_df.loc[(inventory_df['is_used'] == True) & (inventory_df['listing_price'] == 0), 'listing_price'] = np.nan

# Display the count of missing values after cleaning
print("\nAfter cleaning:")
print(f"Missing values in listing_price: {inventory_df['listing_price'].isna().sum()}")
print(f"Missing values in listing_mileage: {inventory_df['listing_mileage'].isna().sum()}")

Before cleaning:
Missing values in listing_price: 498390
Missing values in listing_mileage: 895926

After cleaning:
Missing values in listing_price: 498390
Missing values in listing_mileage: 895927


In [36]:
import sqlalchemy
from sqlalchemy import create_engine

# Create a connection to PostgreSQL
# Replace these parameters with your actual database connection details
username = 'panchito'
password = 'my12$00+Manzanas'
host = '158.120.255.234'
port = '5432'
database ='cars_inventory'

# Create the connection string
conn_string = f"postgresql://{username}:{password}@{host}:{port}/{database}"

# Create the SQLAlchemy engine
engine = create_engine(conn_string)

# Write the DataFrame to PostgreSQL
inventory_df.to_sql(
  name='vehicle_inventory',  # table name
  con=engine,
  if_exists='replace',  # options: 'fail', 'replace', 'append'
  index=False,
  schema='public',  # default schema
  chunksize=1000,  # number of rows to send to the database at once
)

print(f"Data successfully written to PostgreSQL table 'vehicle_inventory'")

# Close the connection
engine.dispose()

Data successfully written to PostgreSQL table 'vehicle_inventory'
